In [ ]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# 不对运行中的Dataset热替换算子：发现旧内核时停止，重启后再读取checkpoint。
def _assert_current_kernel():
    import os
    try:
        shell = get_ipython()
    except NameError:
        return  # CLI是独立Python进程。
    if shell is None or not hasattr(shell, 'kernel'):
        return
    boot = next(int(line.split()[1]) for line in Path('/proc/stat').read_text().splitlines() if line.startswith('btime '))
    started = boot + int(Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    changed = []
    for name, module in tuple(sys.modules.items()):
        if name.startswith(('curation.', 'demiflow.')):
            filename = getattr(module, '__file__', None)
            if filename and Path(filename).is_file() and Path(filename).stat().st_mtime > started:
                changed.append(name)
    if changed:
        raise RuntimeError('当前内核启动后算子代码已更新，请重启内核并重新执行初始化；禁止新旧算子混用。涉及：' + ', '.join(changed[:6]))
_assert_current_kernel()
from curation.v4.ops.filter_document_blocks import FilterDocumentBlocks
from curation.v4.ops.select_source_records import SelectSourceRecords
from curation.v4.ops.cross_batch import BatchRelationshipReviews, ApplyRelationshipReviews
from demiflow.standalone import local_data
from curation.v4.contracts import snapshot, immutable, digest, source_code, runtime_version
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.image_filter import (IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection)
from curation.v4.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy, validate_material_reuse
from curation.v4.local_review_service import image_review_service
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks,
    ReadDocument, CleanDocument, CheckImage, CountMaterial, NestMaterial,
    merge_concept, distinct, fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import PrepareIdentity, ApplyIdentity
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
from curation.v4.ops.source_blocks import BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection, merge_block_decisions
from curation.v4.ops.multimodal import SelectAvailableImages, BatchImageSelection, ApplyImageSelection, merge_image_decisions, SelectRelatedMaterials
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch, ParagraphRows
from curation.v4.ops.paragraphs import ApplyParagraphs, ApplyParagraphReview, SelectRetainedParagraphs
from curation.v4.ops.token_routing import RouteByTokenBudget
from curation.v4.ops.paragraph_pipeline import RouteConceptMaterials, PrepareVerifiedParagraphs, BuildLocalMergeGroups, ApplyLocalIntegration, SourceCatalog, FormatTopicArticle, FinalKnowledgeRecord
from curation.v4.ops.cross_batch import PlanCrossBatchReview, ApplyCrossBatchReview
from curation.v4.ops.paragraph_merge import ApplyParagraphMerge
from curation.v4.ops.topic_quality import PrepareTopicRepairs, ApplyTopicRepairs, RetainReviewedTopics, PrepareTopicVerification
from curation.v4.ops.topic_articles import TopicRows

DATASET = ROOT / 'datasets/demiwtg'
RUN = ROOT / 'state/curation/v4/glass_operator_dual_image_v1'
CONCEPT = '玻璃棒'
IDS = ['legacy:' + CONCEPT]
GROUP_SIZE = 256
# 本轮三个概念的明确身份范围；扩量时从概念资料确定，勿按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None,
          'max_output_tokens':16384, 'temperature':0, 'timeout_s':900,
          'block_unit_chars':1800, 'block_batch_chars':8000,
          'joint_input_tokens':32768, 'image_batch_size':4,
          'text_embedding_model':str(ROOT.parent / 'models/Qwen3-Embedding-0.6B'),
          'image_embedding_model':str(ROOT.parent / 'models/siglip2-base-patch16-224')}
tables = RUN / 'datasets'
knowledge_run = RUN / 'knowledge'

def show(ds, columns=None, n=100):
    from curation.notebook_image_preview import show as preview
    return preview(ds, columns=columns, n=n, run=RUN, dataset=DATASET)


In [ ]:
DATASET = ROOT / 'state/curation/notebook_runtime_check_v1/empty_dataset'
RUN = ROOT / 'state/curation/notebook_runtime_check_v1/glass_empty'
tables = RUN / 'datasets'
knowledge_run = RUN / 'knowledge'
show = lambda *args, **kwargs: None

In [ ]:
concept_source = {'kind':'legacy_concepts', **snapshot(DATASET / 'meta/concepts.json')}
document_source = {'kind':'legacy_docs', **snapshot(DATASET / 'meta/docs.jsonl')}
image_source = {'kind':'legacy_images', **snapshot(DATASET / 'meta/images.jsonl')}
notebook = json.loads((ROOT / 'curation/v4/glass_operator_debug.ipynb').read_text())
manifest = {'sources':[concept_source, document_source, image_source],
            'ids':IDS, 'group_size':GROUP_SIZE, 'config':config,
            'code':source_code(), 'runtime':runtime_version(),
            'cells':[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code']}
from curation.notebook_image_preview import preserve_display_version
manifest = preserve_display_version(RUN, manifest)
immutable(RUN / 'manifest.json', manifest)
version = digest(manifest)
pack, prompt_text = knowledge_prompt_pack(config)
options = prompt_execution_options(RUN, config)
save_prompt_config(RUN, prompt_text, options)
data = local_data(prompt_packs={'knowledge.yaml':pack}, prompt_options=options)

# 每个后续步骤执行前核对运行中代码/依赖/配置，禁止沿用旧version写新结果。
import copy
_frozen_code, _frozen_runtime = source_code(), runtime_version()
_frozen_config, _frozen_run = copy.deepcopy(config), RUN

def _assert_run_current():
    _assert_current_kernel()
    if RUN != _frozen_run or config != _frozen_config or source_code() != _frozen_code or runtime_version() != _frozen_runtime:
        raise RuntimeError('本次冻结后代码、依赖、配置或RUN已变化；请重启内核并使用新RUN，不得混用旧checkpoint。')


In [ ]:
_assert_run_current()
concept_records = data.read_records(DATASET / 'meta/concepts.json', format='json', item_prefix='concepts.item',
    report_path=RUN / 'source_status/concepts.json').filter(SelectSourceRecords('legacy_concepts', IDS))
# 原生读取后下推概念筛选；仍扫描原清单，只有入选关联记录进入后续转换。
show(concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict) and r['value'].get('name') == CONCEPT), n=1)


In [ ]:
_assert_run_current()
concepts = concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict)).map(ConceptFromRecord(concept_source))
show(concepts.filter(lambda r:r['concept_ref'] in IDS), n=1)


In [ ]:
_assert_run_current()
selected_concepts = await concepts.map(SelectConcept(IDS)).filter(lambda r: r['selected']).reduce_by_key('concept_ref', merge_concept).checkpoint_async(tables / 'selected_concepts.jsonl', version=version)
show(selected_concepts, columns=['concept_ref', 'name', 'aliases', 'qid', 'source_records'])


In [ ]:
_assert_run_current()
document_records = data.read_records(DATASET / 'meta/docs.jsonl', report_path=RUN / 'source_status/documents.json').filter(SelectSourceRecords('legacy_docs', IDS))
show(document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in r['value'].get('concepts', [])), n=5)


In [ ]:
_assert_run_current()
documents = document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(DocumentFromRecord(document_source))
show(documents.filter(lambda r: any(ref in IDS for ref in r['concept_refs'])), columns=['doc_id','title','url','path','concept_refs'])


In [ ]:
_assert_run_current()
document_links = await documents.flat_map(MaterialLinks('doc_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'document_links.jsonl', version=version)
show(document_links)


In [ ]:
_assert_run_current()
selected_documents = await documents.join(document_links.select_columns(['doc_id']).reduce_by_key('doc_id', distinct), on='doc_id', how='semi').checkpoint_async(tables / 'selected_documents.jsonl', version=version)
show(selected_documents, columns=['doc_id', 'title', 'url', 'path'])


In [ ]:
_assert_run_current()
raw_documents = await selected_documents.map_cached(ReadDocument(DATASET), cache_dir=RUN / 'cache/read_documents', version=version).checkpoint_async(tables / 'raw_documents.jsonl', version=version)
show(raw_documents, columns=['doc_id', 'title', 'url', 'raw_text', 'read_status', 'read_error'])


In [ ]:
_assert_run_current()
baseline_documents = await raw_documents.map_cached(CleanDocument(), cache_dir=RUN / 'cache/clean_documents', version=version).checkpoint_async(tables / 'baseline_documents.jsonl', version=version)
show(baseline_documents, columns=['doc_id', 'title', 'raw_text', 'clean_text', 'clean_blocks', 'clean_status'])


In [ ]:
_assert_run_current()
processed_documents = await baseline_documents.map_cached(FilterDocumentBlocks(), cache_dir=RUN / 'cache/filter_documents', version=version).checkpoint_async(tables / 'processed_documents.jsonl', version=version)
show(processed_documents, columns=['doc_id', 'title', 'clean_text', 'clean_blocks', 'clean_filter', 'knowledge_eligibility'])


In [ ]:
_assert_run_current()
image_records = data.read_records(DATASET / 'meta/images.jsonl', report_path=RUN / 'source_status/images.json').filter(SelectSourceRecords('legacy_images', IDS))
show(image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in (r['value'].get('concepts') or r['value'].get('instances') or [])), n=5)


In [ ]:
_assert_run_current()
images = image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(ImageFromRecord(image_source))
show(images.filter(lambda r:any(ref in IDS for ref in r['concept_refs'])), columns=['image_id','path','caption','concept_refs'], n=5)


In [ ]:
_assert_run_current()
image_links = await images.flat_map(MaterialLinks('image_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'image_links.jsonl', version=version)
show(image_links)


In [ ]:
_assert_run_current()
selected_images = await images.join(image_links.select_columns(['image_id']).reduce_by_key('image_id', distinct), on='image_id', how='semi').checkpoint_async(tables / 'selected_images.jsonl', version=version)
show(selected_images, columns=['image_id', 'path', 'caption', 'content_url', 'landing_url'])


In [ ]:
_assert_run_current()
processed_images = await selected_images.map_cached(CheckImage(DATASET), cache_dir=RUN / 'cache/check_images', version=version).checkpoint_async(tables / 'processed_images.jsonl', version=version)
show(processed_images, columns=['image_id', 'path', 'byte_status', 'byte_details'])


In [ ]:
_assert_run_current()
document_counts = await document_links.join(processed_documents.select_columns(['doc_id', 'read_status']), on='doc_id').reduce_by_key('concept_ref', CountMaterial('document_count', 'read_status', 'readable_documents')).checkpoint_async(tables / 'document_counts.jsonl', version=version)
show(document_counts)


In [ ]:
_assert_run_current()
image_counts = await image_links.join(processed_images.select_columns(['image_id', 'byte_status']), on='image_id').reduce_by_key('concept_ref', CountMaterial('image_count', 'byte_status', 'verified_images')).checkpoint_async(tables / 'image_counts.jsonl', version=version)
show(image_counts)


In [ ]:
_assert_run_current()
concepts_ready = await selected_concepts.join(document_counts, on='concept_ref', how='left').join(image_counts, on='concept_ref', how='left').map(fill_material_counts).checkpoint_async(tables / 'concepts_ready.jsonl', version=version)
show(concepts_ready, columns=['concept_ref', 'name', 'document_count', 'readable_documents', 'image_count', 'verified_images'])


In [ ]:
_assert_run_current()
concept_documents = await document_links.join(processed_documents.map(NestMaterial('doc_id', 'documents')), on='doc_id').checkpoint_async(tables / 'concept_documents.jsonl', version=version)
show(concept_documents)


In [ ]:
_assert_run_current()
concept_images = await image_links.join(processed_images.map(NestMaterial('image_id', 'images')), on='image_id').checkpoint_async(tables / 'concept_images.jsonl', version=version)
show(concept_images)


In [ ]:
_assert_run_current()
material_batches = await concept_documents.union(concept_images).group_batches('concept_ref', max_rows=GROUP_SIZE, output='materials').checkpoint_async(tables / 'material_batches.jsonl', version=version)
show(material_batches)


In [ ]:
_assert_run_current()
batches = await concepts_ready.join(material_batches, on='concept_ref', how='left').checkpoint_async(tables / 'batches.jsonl', version=version)
show(batches, columns=['concept_ref', 'name', 'materials'])


In [ ]:
_assert_run_current()
model_inputs = await batches.map(model_input).checkpoint_async(tables / 'model_inputs.jsonl', version=version)
show(model_inputs)


In [ ]:
_assert_run_current()
identity_inputs = await model_inputs.map_cached(PrepareIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_prepare', version=version).checkpoint_async(tables / 'identity_inputs.jsonl', version=version)
show(identity_inputs, columns=['case_id', 'identity_prompt'])


In [ ]:
_assert_run_current()
identity_responses = await identity_inputs.map_prompt_async('identity', config='knowledge.yaml', inputs={'payload': 'identity_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1, when=lambda r: not r.get('blocked') and 'identity_prompt' in r).checkpoint_async(tables / 'identity_responses.jsonl', version=version)
show(identity_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
identified = await identity_responses.map_cached(ApplyIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_apply', version=version).checkpoint_async(tables / 'identified.jsonl', version=version)
show(identified, columns=['case_id', 'identity', 'identity_materials', 'identity_unexamined'])


In [ ]:
_assert_run_current()
source_blocks = await identified.map(BuildSourceBlocks(config['block_unit_chars'], body_only=True)).checkpoint_async(tables / 'source_blocks.jsonl', version=version)
show(source_blocks, columns=['case_id', 'source_units'])


In [ ]:
_assert_run_current()
blocks = await source_blocks.map(SelectAvailableImages()).checkpoint_async(tables / 'blocks.jsonl', version=version)
show(blocks, columns=['case_id', 'available_images', 'image_material_scope'])


In [ ]:
_assert_run_current()
text_requests = await blocks.flat_map(BatchSourceBlocks(config['block_batch_chars'])).checkpoint_async(tables / 'text_requests.jsonl', version=version)
show(text_requests, columns=['case_id', 'block_prompt'])


In [ ]:
_assert_run_current()
text_responses = await text_requests.map_prompt_async('select_blocks', config='knowledge.yaml', inputs={'payload': 'block_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'text_responses.jsonl', version=version)
show(text_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
text_decisions = await text_responses.map_cached(ApplyBlockSelection(relevance_only=True), cache_dir=RUN / 'cache/text_selection', version=version).checkpoint_async(tables / 'text_decisions.jsonl', version=version)
show(text_decisions)


In [ ]:
_assert_run_current()
text_by_concept = await text_decisions.reduce_by_key('case_id', merge_block_decisions).checkpoint_async(tables / 'text_by_concept.jsonl', version=version)
show(text_by_concept)


In [ ]:
_assert_run_current()
image_requests = await blocks.flat_map(BatchImageSelection(config['image_batch_size'], config['image_identity_definitions'], neutral=True)).checkpoint_async(tables / 'image_requests.jsonl', version=version)
show(image_requests, columns=['case_id', 'image_prompt', 'pixel_images'])


In [ ]:
_assert_run_current()
primary_data = image_prompt_data(RUN, config)
image_responses = await primary_data.read_json(str(tables / 'image_requests.jsonl')).map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload': 'image_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'image_responses.jsonl', version=version)
show(image_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
# 输入：Qwen响应及原批次像素。输出：初筛判断及独立复核请求；最终判断见34C。
primary = await image_responses.map_cached(RecordPrimaryImageSelection(), cache_dir=RUN / 'cache/image_primary', version=version).checkpoint_async(tables / 'image_primary.jsonl', version=version)
review_rows = await primary.map(PrepareImageReview()).checkpoint_async(tables / 'image_review_inputs.jsonl', version=version)
show(review_rows, columns=['image_prompt', 'review_required', 'primary_selection'])

In [ ]:
_assert_run_current()
review_data = image_prompt_data(RUN, config, review=True)
review_requests = review_data.read_json(str(tables / 'image_review_inputs.jsonl')).filter(lambda r: r['review_required'])
review_path = tables / 'image_review_responses.jsonl'
with image_review_service(RUN, config, needed=review_needed(review_requests, review_path, version)):
    reviewed = await review_requests.map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload':'image_prompt','images':'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=config['image_review_concurrency'], queue_depth=config['image_review_concurrency']).checkpoint_async(review_path, version=version)
show(reviewed, columns=['image_prompt', 'prompt_result', 'prompt_error'])

In [ ]:
_assert_run_current()
image_decisions = await reviewed.union(review_rows.filter(lambda r: not r['review_required'])).map_cached(ApplyConfirmedImageSelection(), cache_dir=RUN / 'cache/image_confirmed', version=version).checkpoint_async(tables / 'image_decisions.jsonl', version=version)
show(image_decisions, columns=['case_id', 'image_decisions'])

In [ ]:
_assert_run_current()
image_by_concept = await image_decisions.reduce_by_key('case_id', merge_image_decisions).checkpoint_async(tables / 'image_by_concept.jsonl', version=version)
show(image_by_concept)


In [ ]:
_assert_run_current()
related = await blocks.join(text_by_concept, on='case_id', how='left').join(image_by_concept, on='case_id', how='left').map(SelectRelatedMaterials()).checkpoint_async(tables / 'related.jsonl', version=version)
show(related, columns=['case_id', 'material_pack'])

save_image_filter_policy(RUN, config)

In [ ]:
_assert_run_current()
routing_materials = await related.map(PrepareRoutingMaterials()).checkpoint_async(tables / 'routing_materials.jsonl', version=version)
show(routing_materials, columns=['concept', 'passages', 'images', 'native_links', 'unmatched_native_references'])


In [ ]:
_assert_run_current()
passage_rows = await routing_materials.flat_map(RawPassageRows()).checkpoint_async(tables / 'passage_rows.jsonl', version=version)
show(passage_rows)


In [ ]:
_assert_run_current()
text_embeddings = await passage_rows.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/input_embeddings', version=version).checkpoint_async(tables / 'text_embeddings.jsonl', version=version)
show(text_embeddings)


In [ ]:
_assert_run_current()
text_vectors = await text_embeddings.flat_map(lambda r: r['items']).reduce_by_key('case_id', lambda a, r: {'case_id': r['case_id'], 'passage_embeddings': {**a['passage_embeddings'], r['source_id']: r}}, initial={'passage_embeddings': {}}).checkpoint_async(tables / 'text_vectors.jsonl', version=version)
show(text_vectors)


In [ ]:
_assert_run_current()
image_text_vectors = await routing_materials.map_cached(EncodeImageTextMaterials(config['image_embedding_model']), cache_dir=RUN / 'cache/input_image_embeddings', version=version).checkpoint_async(tables / 'image_text_vectors.jsonl', version=version)
show(image_text_vectors)


In [ ]:
_assert_run_current()
routed = await routing_materials.join(text_vectors, on='case_id', how='left').join(image_text_vectors.select_columns(['case_id', 'text_windows', 'image_vectors']), on='case_id', how='left').map(RouteByTokenBudget(ROOT.parent/'models/Qwen3.8-27B', config.get('joint_input_tokens',32768))).checkpoint_async(tables / 'routed.jsonl', version=version)
show(routed, columns=['concept', 'requests', 'edges', 'overflow_edges'])


In [ ]:
_assert_run_current()
joint_requests = await routed.flat_map(lambda r: r['requests']).map(BuildRoutedJointRequest()).checkpoint_async(tables / 'joint_requests.jsonl', version=version)
show(joint_requests, columns=['batch_id', 'joint_prompt', 'pixel_images'])


In [ ]:
_assert_run_current()
joint_responses = await joint_requests.map_prompt_async('joint_paragraphs', config='knowledge.yaml', inputs={'payload': 'joint_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'joint_responses.jsonl', version=version)
show(joint_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
extracted = await joint_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/paragraph_extract', version=version).checkpoint_async(tables / 'extracted.jsonl', version=version)
show(extracted, columns=['batch_id', 'topics', 'coverage_note'])


In [ ]:
_assert_run_current()
verify_responses = await extracted.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'verify_responses.jsonl', version=version)
show(verify_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
verified = await verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/paragraph_verify', version=version).checkpoint_async(tables / 'verified.jsonl', version=version)
show(verified, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
originals = await verified.join(joint_requests.select_columns(['batch_id', 'pixel_images']), on='batch_id', how='left').map(PrepareVerifiedParagraphs(RUN)).checkpoint_async(tables / 'originals.jsonl', version=version)
show(originals, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
initial_repair_requests = await originals.flat_map(PrepareTopicRepairs()).checkpoint_async(tables / 'initial_repair_requests.jsonl', version=version)
show(initial_repair_requests, columns=['batch_id', 'repair_payload', 'pixel_images'])


In [ ]:
_assert_run_current()
initial_repair_responses = await initial_repair_requests.map_prompt_async('repair_topics', config='knowledge.yaml', inputs={'payload': 'repair_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'initial_repair_responses.jsonl', version=version)
show(initial_repair_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
initial_repair_extracted = await initial_repair_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/initial_repair_extract', version=version).checkpoint_async(tables / 'initial_repair_extracted.jsonl', version=version)
show(initial_repair_extracted, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
initial_repair_verify_inputs = await initial_repair_extracted.map(PrepareTopicVerification()).checkpoint_async(tables / 'initial_repair_verify_inputs.jsonl', version=version)
show(initial_repair_verify_inputs, columns=['batch_id', 'verify_payload', 'pixel_images'])


In [ ]:
_assert_run_current()
initial_repair_verify_responses = await initial_repair_verify_inputs.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'initial_repair_verify_responses.jsonl', version=version)
show(initial_repair_verify_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
initial_repair_checked = await initial_repair_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/initial_repair_verify', version=version).checkpoint_async(tables / 'initial_repair_checked.jsonl', version=version)
show(initial_repair_checked, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
initial_repair_with_pixels = initial_repair_checked.join(initial_repair_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
initial_repair_results = initial_repair_with_pixels.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repairs':a['repairs']+[r]},initial={'repairs':[]})
initial_repair_planned = initial_repair_requests.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repair_requests':a['repair_requests']+[{'parent_topic_index':r['parent_topic_index']}]},initial={'repair_requests':[]})
show(initial_repair_planned)


In [ ]:
_assert_run_current()
initial_repair_assembled = await originals.join(initial_repair_results, on='batch_id', how='left').join(initial_repair_planned, on='batch_id', how='left').map(ApplyTopicRepairs()).checkpoint_async(tables / 'initial_repair_assembled.jsonl', version=version)
show(initial_repair_assembled)


In [ ]:
_assert_run_current()
initial_repair_topics = await initial_repair_assembled.flat_map(lambda r: r['rows']).map(RetainReviewedTopics()).checkpoint_async(tables / 'initial_repair_topics.jsonl', version=version)
show(initial_repair_topics, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
round_1_retained = await initial_repair_topics.map(SelectRetainedParagraphs()).checkpoint_async(tables / 'round_1_retained.jsonl', version=version)
show(round_1_retained, columns=['content'])


In [ ]:
_assert_run_current()
round_1_paragraphs = await round_1_retained.map(lambda r: r['content']).flat_map(ParagraphRows()).checkpoint_async(tables / 'round_1_paragraphs.jsonl', version=version)
show(round_1_paragraphs)


In [ ]:
_assert_run_current()
round_1_embeddings = await round_1_paragraphs.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/round_1_embeddings', version=version).checkpoint_async(tables / 'round_1_embeddings.jsonl', version=version)
show(round_1_embeddings)


In [ ]:
_assert_run_current()
round_1_groups = await round_1_embeddings.flat_map(lambda r: r['items']).reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'items': a['items'] + [r]}, initial={'items': []}).checkpoint_async(tables / 'round_1_groups.jsonl', version=version)
show(round_1_groups)


In [ ]:
_assert_run_current()
round_1_plans = await round_1_groups.map(PlanCrossBatchReview(include_same_batch=True, skip_single_batch=True)).checkpoint_async(tables / 'round_1_plans.jsonl', version=version)
show(round_1_plans)


In [ ]:
_assert_run_current()
round_1_pairs = await round_1_plans.flat_map(BatchRelationshipReviews()).checkpoint_async(tables / 'round_1_pairs.jsonl', version=version)
show(round_1_pairs, columns=['batch_id', 'review_payload'])


In [ ]:
_assert_run_current()
round_1_responses = await round_1_pairs.map_prompt_async('review_relationships', config='knowledge.yaml', inputs={'payload': 'review_payload'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', when=lambda r:not r.get('batch_error'), concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_responses.jsonl', version=version)
show(round_1_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
round_1_review_batches = await round_1_responses.map_cached(ApplyRelationshipReviews(), cache_dir=RUN / 'cache/round_1_relations', version=version).checkpoint_async(tables / 'round_1_review_batches.jsonl', version=version)
round_1_reviews = await round_1_review_batches.flat_map(lambda r:r['pair_reviews']).checkpoint_async(tables / 'round_1_reviews.jsonl', version=version)
show(round_1_reviews, columns=['paragraph_ids', 'relationship_review', 'next_action'])


In [ ]:
_assert_run_current()
round_1_sources = initial_repair_topics.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'source_rows':a['source_rows']+[r]},initial={'source_rows':[]})
round_1_relations = round_1_reviews.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'review_rows':a['review_rows']+[r]},initial={'review_rows':[]})
round_1_inputs = round_1_groups.join(round_1_relations,on='concept',how='left').join(round_1_sources,on='concept',how='left')
show(round_1_inputs, columns=['concept','review_rows'])


In [ ]:
_assert_run_current()
round_1_merge_plan = await round_1_inputs.map(BuildLocalMergeGroups()).checkpoint_async(tables / 'round_1_merge_plan.jsonl', version=version)
show(round_1_merge_plan)


In [ ]:
_assert_run_current()
round_1_merge_requests = await round_1_merge_plan.flat_map(lambda r: r['requests']).checkpoint_async(tables / 'round_1_merge_requests.jsonl', version=version)
show(round_1_merge_requests, columns=['batch_id', 'merge_payload', 'pixel_images'])


In [ ]:
_assert_run_current()
round_1_merge_responses = await round_1_merge_requests.map_prompt_async('merge_paragraphs', config='knowledge.yaml', inputs={'payload': 'merge_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_merge_responses.jsonl', version=version)
show(round_1_merge_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
round_1_merge_extracted = await round_1_merge_responses.map_cached(ApplyParagraphMerge(), cache_dir=RUN / 'cache/round_1_merge', version=version).checkpoint_async(tables / 'round_1_merge_extracted.jsonl', version=version)
show(round_1_merge_extracted, columns=['batch_id', 'topics', 'merge_validation_issues'])


In [ ]:
_assert_run_current()
round_1_merge_verify_responses = await round_1_merge_extracted.map_prompt_async('verify_merged_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_merge_verify_responses.jsonl', version=version)
show(round_1_merge_verify_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
round_1_merge_checked = await round_1_merge_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_1_merge_verify', version=version).checkpoint_async(tables / 'round_1_merge_checked.jsonl', version=version)
show(round_1_merge_checked, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
round_1_merged = round_1_merge_checked.join(round_1_merge_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left').map(PrepareVerifiedParagraphs(RUN))
round_1_merged_by_concept = round_1_merged.reduce_by_key('concept',lambda a,r:{'concept':r['concept'],'local_results':a['local_results']+[r]},initial={'local_results':[]})
round_1_assembly_input = round_1_sources.join(round_1_merged_by_concept,on='concept',how='left')
show(round_1_merged, columns=['concept','batch_id','topics'])


In [ ]:
_assert_run_current()
round_1_assembled = await round_1_assembly_input.map(ApplyLocalIntegration()).checkpoint_async(tables / 'round_1_assembled.jsonl', version=version)
show(round_1_assembled)


In [ ]:
_assert_run_current()
round_1_rows = await round_1_assembled.flat_map(lambda r: r['rows']).checkpoint_async(tables / 'round_1_rows.jsonl', version=version)
show(round_1_rows, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
round_1_repair_requests = await round_1_rows.flat_map(PrepareTopicRepairs()).checkpoint_async(tables / 'round_1_repair_requests.jsonl', version=version)
show(round_1_repair_requests, columns=['batch_id', 'repair_payload', 'pixel_images'])


In [ ]:
_assert_run_current()
round_1_repair_responses = await round_1_repair_requests.map_prompt_async('repair_topics', config='knowledge.yaml', inputs={'payload': 'repair_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_repair_responses.jsonl', version=version)
show(round_1_repair_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
round_1_repair_extracted = await round_1_repair_responses.map_cached(ApplyParagraphs(), cache_dir=RUN / 'cache/round_1_repair_extract', version=version).checkpoint_async(tables / 'round_1_repair_extracted.jsonl', version=version)
show(round_1_repair_extracted, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
round_1_repair_verify_inputs = await round_1_repair_extracted.map(PrepareTopicVerification()).checkpoint_async(tables / 'round_1_repair_verify_inputs.jsonl', version=version)
show(round_1_repair_verify_inputs, columns=['batch_id', 'verify_payload', 'pixel_images'])


In [ ]:
_assert_run_current()
round_1_repair_verify_responses = await round_1_repair_verify_inputs.map_prompt_async('verify_paragraphs', config='knowledge.yaml', inputs={'payload': 'verify_payload', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'round_1_repair_verify_responses.jsonl', version=version)
show(round_1_repair_verify_responses, columns=['prompt_result', 'prompt_error'])


In [ ]:
_assert_run_current()
round_1_repair_checked = await round_1_repair_verify_responses.map_cached(ApplyParagraphReview(), cache_dir=RUN / 'cache/round_1_repair_verify', version=version).checkpoint_async(tables / 'round_1_repair_checked.jsonl', version=version)
show(round_1_repair_checked, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
round_1_repair_with_pixels = round_1_repair_checked.join(round_1_repair_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
round_1_repair_results = round_1_repair_with_pixels.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repairs':a['repairs']+[r]},initial={'repairs':[]})
round_1_repair_planned = round_1_repair_requests.reduce_by_key('parent_batch_id',lambda a,r:{'batch_id':r['parent_batch_id'],'repair_requests':a['repair_requests']+[{'parent_topic_index':r['parent_topic_index']}]},initial={'repair_requests':[]})
show(round_1_repair_planned)


In [ ]:
_assert_run_current()
round_1_repair_assembled = await round_1_rows.join(round_1_repair_results, on='batch_id', how='left').join(round_1_repair_planned, on='batch_id', how='left').map(ApplyTopicRepairs()).checkpoint_async(tables / 'round_1_repair_assembled.jsonl', version=version)
show(round_1_repair_assembled)


In [ ]:
_assert_run_current()
round_1_repair_topics = await round_1_repair_assembled.flat_map(lambda r: r['rows']).map(RetainReviewedTopics()).checkpoint_async(tables / 'round_1_repair_topics.jsonl', version=version)
show(round_1_repair_topics, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
final_rows = await round_1_repair_topics.checkpoint_async(RUN / 'paragraphs.jsonl', version=version)
all_requests = joint_requests.union(initial_repair_requests).union(round_1_merge_requests).union(round_1_repair_requests)
await all_requests.checkpoint_async(RUN / 'requests.jsonl', version=version)
show(final_rows, columns=['batch_id', 'topics'])


In [ ]:
_assert_run_current()
catalogs = await related.map(SourceCatalog()).checkpoint_async(tables / 'catalogs.jsonl', version=version)
show(catalogs)


In [ ]:
_assert_run_current()
topic_rows = await final_rows.map(SelectRetainedParagraphs()).map(lambda r: r['content']).flat_map(TopicRows()).checkpoint_async(tables / 'topic_rows.jsonl', version=version)
show(topic_rows)


In [ ]:
_assert_run_current()
articles = await topic_rows.join(catalogs, on='concept', how='left').map(FormatTopicArticle()).checkpoint_async(tables / 'articles.jsonl', version=version)
show(articles, columns=['concept', 'article'])


In [ ]:
_assert_run_current()
concept_articles = articles.reduce_by_key('concept', lambda a, r: {'concept': r['concept'], 'articles': a['articles'] + [r['article']]}, initial={'articles': []})
local_audit = round_1_merge_plan.map(lambda r: {'concept': r['concept'], 'local_audit': {'pending': r['pending'], 'local_task_count': len(r['requests'])}})
plan_audit = round_1_plans.map(lambda r: {'concept': r['concept'], 'cross_batch_plan': {'pending': r['pending'], 'candidate_count': len(r['review_requests']), 'passthrough_paragraph_ids': r['passthrough_paragraph_ids']}})
knowledge = await related.map(lambda r: {**r, 'concept': r['identity']['target_label']}).join(concept_articles, on='concept', how='left').join(local_audit, on='concept', how='left').join(plan_audit, on='concept', how='left').map(FinalKnowledgeRecord()).checkpoint_async(RUN / 'knowledge_base.jsonl', version=version)
show(knowledge, columns=['concept', 'knowledge'])


In [ ]:
_assert_run_current()
from curation.v4.current_results import show_current_results
show_current_results(RUN, concepts=[CONCEPT], limit=10, images=True)

In [ ]:
assert knowledge.take_all() == []
assert not list((RUN / 'knowledge/calls').glob('*.request.json'))
print('PASS: all glass operator cells executed on empty input, zero model calls')